In [1]:
import numpy as np
from scipy.stats import entropy
from sklearn.metrics import roc_auc_score
import pandas as pd 

def AULC(accs, uncertainties):
  idxs = np.argsort(uncertainties)
  uncs_s = uncertainties[idxs]
  error_s = accs[idxs]

  mean_error = error_s.mean()
  error_csum = np.cumsum(error_s)

  Fs = error_csum / np.arange(1, len(error_s) + 1)
  s = 1 / len(Fs)
  return -1 + s * Fs.sum() / mean_error, Fs

def rAULC(uncertainties, accs):
    perf_aulc, Fsp = AULC(accs, -accs.astype("float"))
    curr_aulc, Fsc = AULC(accs, uncertainties)
    return curr_aulc / perf_aulc

def calculate_accuracy(y_true, y_pred):
    return (y_true == y_pred).sum()/len(y_true)


def calculate_auc(y_true, y_probs):
    return  roc_auc_score(y_true, y_probs, multi_class='ovr')


def calculate_ece(y_true, y_probs, n_bins=30):

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_probs, bin_edges) - 1

    ece = 0.0
    for i in range(n_bins):
        bin_mask = bin_indices == i
        if np.sum(bin_mask) == 0:
            continue
        bin_accuracy = y_true[bin_mask].mean() if np.sum(bin_mask) > 0 else 0
        bin_confidence = y_probs[bin_mask].mean()
        ece += np.abs(bin_accuracy - bin_confidence) * np.sum(bin_mask) / len(y_true)

    return ece


def calculate_mean_entropy(y_probs):
    return np.mean(entropy(y_probs, base=2, axis=1))


In [2]:
def calculate_entropy_single(probabilities, base=2):
    """
    단일 확률 분포의 엔트로피 계산.

    Args:
        probabilities (array-like): 확률 분포 (합이 1이어야 함).
        base (int): 로그의 밑 (기본값: 2).

    Returns:
        float: 엔트로피 값.
    """
    probabilities = np.array(probabilities)
    eps = 1e-12  # 로그 0 방지
    probabilities = np.clip(probabilities, eps, 1)  # 확률 값 제한
    log_probs = np.log(probabilities) / np.log(base)  # 로그 변환
    entropy = -np.sum(probabilities * log_probs)  # 엔트로피 계산
    return entropy

In [3]:
import numpy as np

# 3개의 데이터 샘플, 4개의 클래스에 대한 확률 분포
y_probs = np.array([
    [0.7, 0.2, 0.1, 0.0],[0.3, 0.2, 0.1, 0.4]])


In [4]:
sample_entropies = calculate_mean_entropy(y_probs)
sample_entropies

1.5016094970590275

In [20]:

from scipy.stats import entropy

model = "TransMIL"
i = 1
df = pd.read_csv(f"/mnt/e/{model}/vit-ssl-dino-p16/nontext/seed_{i}/all_predictions.csv")

y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)
df['entropy'] = entropy(y_probs, base=2, axis=1)
df['Uncertainty'] = df['distance'] * df['entropy']

In [21]:
df

,Slide name,GT,Pred,distance,Confidence HP,Confidence IP,Confidence LP,Confidence SSL,Confidence TA,Confidence TSA,Confidence TVA+VA,entropy,Uncertainty
0,2022S 0072235050101,HP,HP,0.062108,0.3117,0.1147,0.1147,0.1147,0.1147,0.1147,0.1147,2.674311,0.166096
1,2022S 0069126010101,HP,HP,0.065958,0.3118,0.1147,0.1147,0.1147,0.1147,0.1147,0.1147,2.674212,0.176385
2,2022S 0067850030101,HP,HP,0.062333,0.3117,0.1147,0.1147,0.1147,0.1147,0.1147,0.1147,2.674311,0.166698
3,2022S 0074026050101,HP,HP,0.069770,0.2866,0.1168,0.1168,0.1168,0.1293,0.1168,0.1168,2.707592,0.188908
4,2022S 0072487020101,HP,HP,0.062529,0.3113,0.1147,0.1148,0.1147,0.1149,0.1147,0.1147,2.674842,0.167255
...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,2022S 0261319030101,TSA,TSA,0.056697,0.1147,0.1147,0.1147,0.1147,0.1147,0.3118,0.1147,2.674212,0.151619
343,2022S 0126481020101,TSA,TSA,0.066656,0.1147,0.1148,0.1147,0.1147,0.1147,0.3117,0.1147,2.674356,0.178261
344,2023S 0109064010101,TSA,TSA,0.060785,0.1147,0.1147,0.1147,0.1148,0.1147,0.3117,0.1147,2.674356,0.162561
345,2022S 0226203090101,TSA,TSA,0.064582,0.1147,0.1147,0.1147,0.1147,0.1147,0.3118,0.1147,2.674212,0.172706


In [29]:
def calculate_entropy_single(probabilities, base=2):
    """
    단일 확률 분포의 엔트로피 계산.

    Args:
        probabilities (array-like): 확률 분포 (합이 1이어야 함).
        base (int): 로그의 밑 (기본값: 2).

    Returns:
        float: 엔트로피 값.
    """
    probabilities = np.array(probabilities)
    eps = 1e-12  # 로그 0 방지
    probabilities = np.clip(probabilities, eps, 1)  # 확률 값 제한
    log_probs = np.log(probabilities) / np.log(base)  # 로그 변환
    entropy = -np.sum(probabilities * log_probs)  # 엔트로피 계산
    return entropy

sample_entropies = entropy(y_probs, base=2, axis=1)
sample_entropies

array([1.15677965, 1.84643934])

In [31]:
char_to_num = {
    'HP': 0,
    'IP': 1,
    'LP': 2,
    'SSL': 3,
    'TA': 4,
    'TSA': 5,
    'TVA+VA': 6
}

acc = []
auc = []
ece = []
entropys1 = []
entropys = []

raulc = []
seeds = [1,17,2000]

model = "TransMIL"
p = 1

for i in seeds:

    #저장 excel 파일 열기
    # Open the saved Excel file
    df = pd.read_csv(f"/mnt/e/{model}/vit-ssl-dino-p16/nontext/seed_{i}/all_predictions.csv")

    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)
    df['entropy'] = entropy(y_probs, base=2, axis=1)
    df['Unceratinty'] = df['distance'] * df['entropy']

    #uncertainty 상위 5%씩 제외하기 위한 과정
    #모든 값은 p = 1, 5%제외시 p = 0.95.......
    # Process to exclude the top 5% of uncertainty values
    # All values initially have p = 1, excluding 5% sets p = 0.95...
    threshold = df['Unceratinty'].quantile(p) 
    df = df[df['Unceratinty'] <= threshold].reset_index(drop = True)
    
    # 더 편하게 하기 위해서 숫자 맵핑(training 후 저장없이 바로 하면 필요없을듯 합니다.
    # For convenience, map numbers (this step may not be necessary if done immediately after training without saving)
    df['GT'] = df['GT'].map(char_to_num)
    df['Pred'] = df['Pred'].map(char_to_num)
    
    # excel로 저장하면서 confidence값의 합이 1이 되지 않는 경우가 발생, 따라서 합을 1로 맞춰줘야합니다 
    # 만약 따로 저장해서 불러사용하는 것이 아닌 training 후에 결과를 바로 내는 거면 굳이 필요없을 합니다.  
    # When saving to Excel, the sum of confidence values may not equal 1, so adjustments are needed to ensure the sum is 1.
    # If results are generated immediately after training without saving and reloading, this step may not be necessary.
    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)

    
    
    
    acc.append(calculate_accuracy(df['GT'],df['Pred'])*100)
    auc.append(roc_auc_score(df['GT'],y_probs, multi_class = 'ovo')*100)
    ece.append(calculate_ece(y_probs,df['GT']))
    entropys.append(calculate_mean_entropy(y_probs))
    raulc.append(rAULC(df['Unceratinty'], (df['GT'] == df['Pred']))*100)

print(acc)
print(auc)
print(ece)
print(entropys)
print(raulc)


print("acc : ", f"{np.mean(acc):.2f} ± {np.std(acc):.2f}")
print("auc : ", f"{np.mean(auc):.2f} ± {np.std(auc):.2f}")
print("ece : ", f"{np.mean(ece):.3f} ± {np.std(ece):.3f}")
print("entropy : ", f"{np.mean(entropys):.3f} ± {np.std(entropys):.3f}")
print("ralulc : ", f"{np.mean(raulc):.2f} ± {np.std(raulc):.2f}") 

[89.9135446685879, 92.21902017291066, 91.93083573487031]
[98.65334048785309, 98.93603616735673, 97.78471990795103]
[0.023054755043227664, 0.023054755043227664, 0.023054755043227664]
[2.6836326945614566, 2.683023582362542, 2.680619602344473]
[17.73413603264064, 11.103187033586982, 19.99694041095993]
acc :  91.35 ± 1.03
auc :  98.46 ± 0.49
ece :  0.023 ± 0.000
entropy :  2.682 ± 0.001
ralulc :  16.28 ± 3.77


In [ ]:
char_to_num = {
    'HP': 0,
    'IP': 1,
    'LP': 2,
    'SSL': 3,
    'TA': 4,
    'TSA': 5,
    'TVA+VA': 6
}

acc = []
auc = []
ece = []
entropys1 = []
entropys = []

raulc = []
seeds = [1,17,2000]

model = "TransMIL"
p = 0.95

for i in seeds:

    #저장 excel 파일 열기
    # Open the saved Excel file
    df = pd.read_csv(f"/mnt/e/{model}/vit-ssl-dino-p16/nontext/seed_{i}/all_predictions.csv")

    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)
    df['entropy'] = entropy(y_probs, base=2, axis=1)
    df['Unceratinty'] = df['distance'] * df['entropy']

    #uncertainty 상위 5%씩 제외하기 위한 과정
    #모든 값은 p = 1, 5%제외시 p = 0.95.......
    # Process to exclude the top 5% of uncertainty values
    # All values initially have p = 1, excluding 5% sets p = 0.95...
    threshold = df['Unceratinty'].quantile(p) 
    df = df[df['Unceratinty'] <= threshold].reset_index(drop = True)
    
    # 더 편하게 하기 위해서 숫자 맵핑(training 후 저장없이 바로 하면 필요없을듯 합니다.
    # For convenience, map numbers (this step may not be necessary if done immediately after training without saving)
    df['GT'] = df['GT'].map(char_to_num)
    df['Pred'] = df['Pred'].map(char_to_num)
    
    # excel로 저장하면서 confidence값의 합이 1이 되지 않는 경우가 발생, 따라서 합을 1로 맞춰줘야합니다 
    # 만약 따로 저장해서 불러사용하는 것이 아닌 training 후에 결과를 바로 내는 거면 굳이 필요없을 합니다.  
    # When saving to Excel, the sum of confidence values may not equal 1, so adjustments are needed to ensure the sum is 1.
    # If results are generated immediately after training without saving and reloading, this step may not be necessary.
    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)

    
    
    
    acc.append(calculate_accuracy(df['GT'],df['Pred'])*100)
    auc.append(roc_auc_score(df['GT'],y_probs, multi_class = 'ovo')*100)
    ece.append(calculate_ece(y_probs,df['GT']))
    entropys.append(calculate_mean_entropy(y_probs))
    raulc.append(rAULC(df['Unceratinty'], (df['GT'] == df['Pred']))*100)

print(acc)
print(auc)
print(ece)
print(entropys)
print(raulc)


print("acc : ", f"{np.mean(acc):.2f} ± {np.std(acc):.2f}")
print("auc : ", f"{np.mean(auc):.2f} ± {np.std(auc):.2f}")
print("ece : ", f"{np.mean(ece):.3f} ± {np.std(ece):.3f}")
print("entropy : ", f"{np.mean(entropys):.3f} ± {np.std(entropys):.3f}")
print("ralulc : ", f"{np.mean(raulc):.2f} ± {np.std(raulc):.2f}") 

[90.88145896656535, 92.40121580547113, 92.70516717325228]
[98.85705088864593, 98.94852244191001, 98.24179311978133]
[0.02214502822405558, 0.021710811984368215, 0.02344767694311767]
[2.6833732610218632, 2.6822899243918266, 2.680251427581074]
[0.533848281768222, 0.4679462786939372, 7.624059228367786]
acc :  92.00 ± 0.80
auc :  98.68 ± 0.31
ece :  0.022 ± 0.001
entropy :  2.682 ± 0.001
ralulc :  2.88 ± 3.36


In [36]:
char_to_num = {
    'HP': 0,
    'IP': 1,
    'LP': 2,
    'SSL': 3,
    'TA': 4,
    'TSA': 5,
    'TVA+VA': 6
}

acc = []
auc = []
ece = []
entropys1 = []
entropys = []

raulc = []
seeds = [1,17,2000]

model = "TransMIL"
p = 0.90

for i in seeds:

    #저장 excel 파일 열기
    # Open the saved Excel file
    df = pd.read_csv(f"/mnt/e/{model}/vit-ssl-dino-p16/nontext/seed_{i}/all_predictions.csv")

    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)
    df['entropy'] = entropy(y_probs, base=2, axis=1)
    df['Unceratinty'] = df['distance'] * df['entropy']

    #uncertainty 상위 5%씩 제외하기 위한 과정
    #모든 값은 p = 1, 5%제외시 p = 0.95.......
    # Process to exclude the top 5% of uncertainty values
    # All values initially have p = 1, excluding 5% sets p = 0.95...
    threshold = df['Unceratinty'].quantile(p) 
    df = df[df['Unceratinty'] <= threshold].reset_index(drop = True)
    
    # 더 편하게 하기 위해서 숫자 맵핑(training 후 저장없이 바로 하면 필요없을듯 합니다.
    # For convenience, map numbers (this step may not be necessary if done immediately after training without saving)
    df['GT'] = df['GT'].map(char_to_num)
    df['Pred'] = df['Pred'].map(char_to_num)
    
    # excel로 저장하면서 confidence값의 합이 1이 되지 않는 경우가 발생, 따라서 합을 1로 맞춰줘야합니다 
    # 만약 따로 저장해서 불러사용하는 것이 아닌 training 후에 결과를 바로 내는 거면 굳이 필요없을 합니다.  
    # When saving to Excel, the sum of confidence values may not equal 1, so adjustments are needed to ensure the sum is 1.
    # If results are generated immediately after training without saving and reloading, this step may not be necessary.
    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)

    
    
    
    acc.append(calculate_accuracy(df['GT'],df['Pred'])*100)
    auc.append(roc_auc_score(df['GT'],y_probs, multi_class = 'ovo')*100)
    ece.append(calculate_ece(y_probs,df['GT']))
    entropys.append(calculate_mean_entropy(y_probs))
    raulc.append(rAULC(df['Unceratinty'], (df['GT'] == df['Pred']))*100)

print(acc)
print(auc)
print(ece)
print(entropys)
print(raulc)


print("acc : ", f"{np.mean(acc):.2f} ± {np.std(acc):.2f}")
print("auc : ", f"{np.mean(auc):.2f} ± {np.std(auc):.2f}")
print("ece : ", f"{np.mean(ece):.3f} ± {np.std(ece):.3f}")
print("entropy : ", f"{np.mean(entropys):.3f} ± {np.std(entropys):.3f}")
print("ralulc : ", f"{np.mean(raulc):.2f} ± {np.std(raulc):.2f}") 

[90.7051282051282, 91.98717948717949, 92.94871794871796]
[98.7957264044766, 98.9007472479387, 98.13535450609626]
[0.022435897435897436, 0.02106227106227106, 0.024725274725274724]
[2.682984849782495, 2.6822239790727886, 2.680157563539215]
[2.520311767876352, 6.002980280007057, 4.642171058038344]
acc :  91.88 ± 0.92
auc :  98.61 ± 0.34
ece :  0.023 ± 0.002
entropy :  2.682 ± 0.001
ralulc :  4.39 ± 1.43


In [37]:
char_to_num = {
    'HP': 0,
    'IP': 1,
    'LP': 2,
    'SSL': 3,
    'TA': 4,
    'TSA': 5,
    'TVA+VA': 6
}

acc = []
auc = []
ece = []
entropys1 = []
entropys = []

raulc = []
seeds = [1,17,2000]

model = "TransMIL"
p = 0.85

for i in seeds:

    #저장 excel 파일 열기
    # Open the saved Excel file
    df = pd.read_csv(f"/mnt/e/{model}/vit-ssl-dino-p16/nontext/seed_{i}/all_predictions.csv")

    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)
    df['entropy'] = entropy(y_probs, base=2, axis=1)
    df['Unceratinty'] = df['distance'] * df['entropy']

    #uncertainty 상위 5%씩 제외하기 위한 과정
    #모든 값은 p = 1, 5%제외시 p = 0.95.......
    # Process to exclude the top 5% of uncertainty values
    # All values initially have p = 1, excluding 5% sets p = 0.95...
    threshold = df['Unceratinty'].quantile(p) 
    df = df[df['Unceratinty'] <= threshold].reset_index(drop = True)
    
    # 더 편하게 하기 위해서 숫자 맵핑(training 후 저장없이 바로 하면 필요없을듯 합니다.
    # For convenience, map numbers (this step may not be necessary if done immediately after training without saving)
    df['GT'] = df['GT'].map(char_to_num)
    df['Pred'] = df['Pred'].map(char_to_num)
    
    # excel로 저장하면서 confidence값의 합이 1이 되지 않는 경우가 발생, 따라서 합을 1로 맞춰줘야합니다 
    # 만약 따로 저장해서 불러사용하는 것이 아닌 training 후에 결과를 바로 내는 거면 굳이 필요없을 합니다.  
    # When saving to Excel, the sum of confidence values may not equal 1, so adjustments are needed to ensure the sum is 1.
    # If results are generated immediately after training without saving and reloading, this step may not be necessary.
    y_probs = df[['Confidence HP', 'Confidence IP', 'Confidence LP','Confidence SSL','Confidence TA','Confidence TSA', 'Confidence TVA+VA']].values  # 클래스별 확률
    y_probs = y_probs / np.sum(y_probs, axis=1, keepdims=True)

    
    
    
    acc.append(calculate_accuracy(df['GT'],df['Pred'])*100)
    auc.append(roc_auc_score(df['GT'],y_probs, multi_class = 'ovo')*100)
    ece.append(calculate_ece(y_probs,df['GT']))
    entropys.append(calculate_mean_entropy(y_probs))
    raulc.append(rAULC(df['Unceratinty'], (df['GT'] == df['Pred']))*100)

print(acc)
print(auc)
print(ece)
print(entropys)
print(raulc)


print("acc : ", f"{np.mean(acc):.2f} ± {np.std(acc):.2f}")
print("auc : ", f"{np.mean(auc):.2f} ± {np.std(auc):.2f}")
print("ece : ", f"{np.mean(ece):.3f} ± {np.std(ece):.3f}")
print("entropy : ", f"{np.mean(entropys):.3f} ± {np.std(entropys):.3f}")
print("ralulc : ", f"{np.mean(raulc):.2f} ± {np.std(raulc):.2f}") 

[90.16949152542372, 91.86440677966101, 92.54237288135593]
[98.73789762926667, 98.84459325964033, 97.9038367465211]
[0.02324455205811138, 0.021307506053268765, 0.0261501210653753]
[2.6829016223127082, 2.6823553170326564, 2.6804990545880467]
[8.431912767595087, 7.993436963576987, 10.479227542844695]
acc :  91.53 ± 1.00
auc :  98.50 ± 0.42
ece :  0.024 ± 0.002
entropy :  2.682 ± 0.001
ralulc :  8.97 ± 1.08


In [ ]:
import E:\TissueFeatures\featuresMultiResolution\vit-ssl-dino-p16\x5\256\test\HP\2022S 0067850030101

In [5]:
import pickle

# .pkl 파일 경로 지정
file_path = "/mnt/e/TissueFeatures/featuresMultiResolution/vit-ssl-dino-p16/x5/256/test/HP/2022S 0067850030101/2022S 0067850030101.pkl"

# .pkl 파일 읽기 모드로 열기
with open(file_path, "rb") as file:
    data = pickle.load(file)

# 로드된 데이터 출력
print(data.keys())

dict_keys(['subsite', 'caption', 'features'])
